# LeetCode 1433 — Check If a String Can Break Another String bằng Python

Link đề: https://leetcode.com/problems/check-if-a-string-can-break-another-string/ · Độ khó: **Medium** · Chủ đề: String, Greedy, Sorting (thêm: Counting)

Skeleton LeetCode:

```python
class Solution:
    def checkIfCanBreak(self, s1: str, s2: str) -> bool:
        ...
```

## 1. Mục tiêu học

1. Hiểu định nghĩa "chuỗi `x` **break** chuỗi `y`" (`x[i] ≥ y[i]` với mọi `i`) và việc được **hoán vị tuỳ ý** cả hai chuỗi.
2. Chứng minh **tham lam bằng sắp xếp**: ghép phần tử nhỏ nhất với nhỏ nhất là cách ghép "tốt nhất có thể" ⇒ chỉ cần sort hai chuỗi rồi so từng vị trí.
3. Tận dụng bảng chữ cái nhỏ (26 chữ): **counting sort** `O(n)` và kiểm tra **tổng tiền tố tần suất** `O(n + 26)` mà không cần dựng lại chuỗi.
4. Viết **brute force hoán vị** cho `n` nhỏ để kiểm chứng.
5. Tránh lỗi: chỉ kiểm tra một chiều, so sánh khi chưa sort, trộn chiều theo từng vị trí, dùng `>` thay `≥`.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s1`, `s2` **cùng độ dài** `n`. Kiểm tra xem có **một hoán vị** của `s1` "break" được **một hoán vị** của `s2`, **hoặc ngược lại** (một hoán vị của `s2` break một hoán vị của `s1`).

Chuỗi `x` **break** chuỗi `y` (cùng độ dài `n`) nếu `x[i] >= y[i]` (theo thứ tự bảng chữ cái) với **mọi** `i` từ `0` đến `n − 1`.

**Ví dụ 1:** `s1 = "abc", s2 = "xya"` → `true` (`"ayx"` — hoán vị của `s2` — break `"abc"`).

**Ví dụ 2:** `s1 = "abe", s2 = "acd"` → `false` (không hoán vị nào của bên này break được bên kia).

**Ví dụ 3:** `s1 = "leetcodee", s2 = "interview"` → `true`.

**Ràng buộc:** `s1.length == s2.length == n`, `1 <= n <= 10^5`, chỉ gồm chữ thường tiếng Anh.

In [1]:
import random
import time
from itertools import permutations, accumulate
from collections import Counter
from typing import List

examples = [(("abc", "xya"), True), (("abe", "acd"), False), (("leetcodee", "interview"), True)]
for (s1, s2), want in examples:
    print(f"{s1!r:12} {s2!r:12} -> {want}")

'abc'        'xya'        -> True
'abe'        'acd'        -> False
'leetcodee'  'interview'  -> True


## 3. Kiến thức nền

### 3.1 So sánh ký tự trong Python

Ký tự so sánh theo mã Unicode: `'a' < 'b' < … < 'z'`. `ord(c) - ord('a')` cho chỉ số 0..25.

In [2]:
print("a" <= "b", "z" >= "y", "c" >= "c")
print([ord(c) - ord("a") for c in "abz"])

True True True
[0, 1, 25]


### 3.2 `sorted` trên chuỗi, `zip`, `all`

`sorted("xya")` trả list ký tự đã sắp `['a','x','y']`. `zip(a, b)` ghép từng cặp; `all(...)` đúng khi mọi phần tử đúng (dừng sớm khi gặp sai).

In [3]:
a, b = sorted("abc"), sorted("xya")
print(a, b, list(zip(a, b)))
print(all(x >= y for x, y in zip(b, a)), all(x >= y for x, y in zip(a, b)))

['a', 'b', 'c'] ['a', 'x', 'y'] [('a', 'a'), ('b', 'x'), ('c', 'y')]
True False


### 3.3 `itertools.permutations` (cho brute force)

Sinh mọi hoán vị (`n!` cái) — chỉ dùng với `n` rất nhỏ để kiểm chứng.

In [4]:
print(["".join(p) for p in permutations("abe")])
print("số hoán vị của chuỗi dài 8:", len(list(permutations(range(8)))))

['abe', 'aeb', 'bae', 'bea', 'eab', 'eba']
số hoán vị của chuỗi dài 8: 40320


### 3.4 Đếm tần suất bằng mảng 26 phần tử và tổng tiền tố

`cnt[k]` = số lần chữ thứ `k` xuất hiện. Tổng tiền tố `pre[k] = cnt[0] + … + cnt[k]` = **số ký tự ≤ chữ thứ `k`**. `itertools.accumulate` tính nhanh.

In [5]:
def count26(s):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    return cnt

c = count26("abcab")
print(c[:5], list(accumulate(c))[:5])

[2, 2, 1, 0, 0] [2, 4, 5, 5, 5]


## 4. Ý tưởng nền tảng / trực giác

Vì được hoán vị **cả hai** chuỗi, thực chất ta cần **ghép cặp** mỗi ký tự của `s1` với đúng một ký tự của `s2` sao cho bên `s1` luôn ≥ (hoặc bên `s2` luôn ≥). Thứ tự các cặp không quan trọng.

Thử tay ví dụ 1: `s1 = "abc"`, `s2 = "xya"`.
- Muốn `s2` break `s1`: ký tự **nhỏ nhất** của `s1` là `a` cần được ghép với một ký tự `≥ a` của `s2`. Ký tự **lớn nhất** của `s1` là `c` cần ký tự `≥ c`. Nên ghép "nhỏ với nhỏ, lớn với lớn": sort `s1 = abc`, `s2 = axy` ⇒ `a≥a`, `x≥b`, `y≥c` ✔ ⇒ `true`.

Ví dụ 2: sort `s1 = abe`, `s2 = acd`:
- `s1 ≥ s2`? `a≥a` ✔, `b≥c` ✘.
- `s2 ≥ s1`? `a≥a` ✔, `c≥b` ✔, `d≥e` ✘.
⇒ `false`.

Trực giác: nếu ngay cả cách ghép "đã sắp xếp" cũng thất bại thì không cách ghép nào thành công (sẽ chứng minh ở mục 10).

In [6]:
for (s1, s2), want in examples:
    a, b = sorted(s1), sorted(s2)
    print("".join(a), "".join(b), "| s1>=s2:", all(x >= y for x, y in zip(a, b)),
          "| s2>=s1:", all(y >= x for x, y in zip(a, b)), "| mong đợi", want)

abc axy | s1>=s2: False | s2>=s1: True | mong đợi True
abe acd | s1>=s2: False | s2>=s1: False | mong đợi False
cdeeeelot eeiinrtvw | s1>=s2: False | s2>=s1: True | mong đợi True


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị

**Ý tưởng:** giữ nguyên `s1` (vì hoán vị đồng thời hai chuỗi theo cùng một cách không đổi kết quả, chỉ cần hoán vị một bên). Thử mọi hoán vị `p` của `s2`: nếu `p` break `s1` hoặc `s1` break `p` ⇒ `True`.

Lưu ý: "một hoán vị của `s1` break một hoán vị của `s2`" ⇔ "tồn tại hoán vị `p` của `s2` sao cho `s1[i] ≥ p[i]` với mọi `i`" (cố định thứ tự `s1`, sắp lại `s2` tương ứng).

In [7]:
def breaks(x, y):
    return all(a >= b for a, b in zip(x, y))

def can_break_brute(s1, s2):
    for p in set(permutations(s2)):
        if breaks(s1, p) or breaks(p, s1):
            return True
    return False

for (s1, s2), want in examples[:2]:
    print(s1, s2, can_break_brute(s1, s2), "✔" if can_break_brute(s1, s2) == want else "✘")

abc xya True ✔
abe acd False ✔


**Độ phức tạp:** `O(n! · n)` — `n = 10` đã là 36 triệu thao tác, `n = 10^5` là không tưởng. Chỉ dùng để kiểm chứng với `n ≤ 7`.

**Nút thắt:** thử vô số cách ghép trong khi chỉ cần **một** cách ghép "tốt nhất" — cách ghép đã sắp xếp.

In [8]:
t = time.perf_counter(); can_break_brute("abcdefg", "gfedcbz"); el = time.perf_counter() - t
print(f"brute n=7: {el:.3f}s  (n=10 sẽ chậm hơn ~720 lần)")

brute n=7: 0.002s  (n=10 sẽ chậm hơn ~720 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — sắp xếp là cách ghép tối ưu:** gọi `a = sorted(s1)`, `b = sorted(s2)`. Khi đó **`s1` có thể break `s2` ⇔ `a[i] ≥ b[i]` với mọi `i`**.
- (⇐) hiển nhiên: dùng chính hoán vị đã sắp.
- (⇒) nếu có cách ghép nào đó thành công, thì với mỗi `i`, phần tử nhỏ thứ `i` của `s1` (`a[i]`)… Lập luận đếm: có `n − i` phần tử của `s1` `≥ a[i]`… (chứng minh đầy đủ ở mục 10).

**Insight 2 — kiểm tra cả hai chiều:** đáp án = `breaks(a, b) or breaks(b, a)`.

**Insight 3 — điều kiện tần suất (không cần sort):** `s1` break `s2` ⇔ với **mọi** chữ `c`, *số ký tự `≤ c` trong `s1`* **≤** *số ký tự `≤ c` trong `s2`*. (Bên "lớn hơn" phải có ít ký tự nhỏ hơn.) Kiểm tra 26 tổng tiền tố ⇒ `O(n + 26)`.

**Insight 4 — một lượt cho cả hai chiều:** duyệt `k = 0..25`, theo dõi `d = pre1[k] − pre2[k]`. `s1` break `s2` ⇔ mọi `d ≤ 0`; `s2` break `s1` ⇔ mọi `d ≥ 0`. Đáp án sai ⇔ `d` đổi dấu (có cả âm lẫn dương).

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp hai chuỗi, so từng vị trí

**Từng đoạn:**
1. `a, b = sorted(s1), sorted(s2)`.
2. `return all(x >= y for x, y in zip(a, b)) or all(y >= x for x, y in zip(a, b))`.

In [9]:
def can_break_sort(s1, s2):
    a, b = sorted(s1), sorted(s2)
    return all(x >= y for x, y in zip(a, b)) or all(y >= x for x, y in zip(a, b))

for (s1, s2), want in examples:
    print(s1, s2, can_break_sort(s1, s2), can_break_sort(s1, s2) == want)

abc xya True True
abe acd False True
leetcodee interview True True


In [10]:
class SolutionSort:
    def checkIfCanBreak(self, s1: str, s2: str) -> bool:
        a, b = sorted(s1), sorted(s2)
        return all(x >= y for x, y in zip(a, b)) or all(y >= x for x, y in zip(a, b))

print([SolutionSort().checkIfCanBreak(*p) for p, _ in examples])

[True, False, True]


**Độ phức tạp:** `O(n log n)` thời gian (sort), `O(n)` bộ nhớ.

### Cách 3 — Counting sort (26 chữ) rồi so như cách 2

**Từng đoạn:**
1. Đếm `cnt1`, `cnt2` (26 phần tử).
2. Dựng lại chuỗi đã sắp: `"".join(chr(97 + k) * cnt[k] for k in range(26))` — `O(n)`.
3. So sánh như cách 2.

Tận dụng bảng chữ cái nhỏ để sort `O(n + 26)` thay vì `O(n log n)`.

In [11]:
def counting_sorted(s):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    return "".join(chr(97 + k) * cnt[k] for k in range(26))

def can_break_counting_sort(s1, s2):
    a, b = counting_sorted(s1), counting_sorted(s2)
    ge = le = True
    for x, y in zip(a, b):
        if x < y: ge = False
        if x > y: le = False
        if not ge and not le:
            return False
    return True

print(counting_sorted("leetcodee"), counting_sorted("interview"))
for (s1, s2), want in examples:
    print(s1, s2, can_break_counting_sort(s1, s2), can_break_counting_sort(s1, s2) == want)

cdeeeelot eeiinrtvw
abc xya True True
abe acd False True
leetcodee interview True True


**Độ phức tạp:** `O(n + 26)` thời gian, `O(n)` bộ nhớ (chuỗi dựng lại). Nhờ cờ `ge/le`, chỉ cần **một lượt** cho cả hai chiều và dừng sớm.

### Cách 4 — Tổng tiền tố tần suất (không dựng lại chuỗi)

**Từng đoạn:**
1. `cnt1`, `cnt2` đếm 26 chữ.
2. Duyệt `k = 0..25`, cộng dồn `d += cnt1[k] − cnt2[k]` (= `pre1[k] − pre2[k]`).
3. Ghi nhận đã thấy `d > 0` (`pos`) hay `d < 0` (`neg`).
4. Đáp án: `not (pos and neg)`.

Ý nghĩa: `d < 0` ở chữ `c` nghĩa là `s2` có **nhiều** ký tự `≤ c` hơn `s1` ⇒ `s2` không thể break `s1`; `d > 0` ⇒ `s1` không thể break `s2`.

In [12]:
def can_break_prefix(s1, s2):
    cnt = [0] * 26
    for x, y in zip(s1, s2):
        cnt[ord(x) - 97] += 1
        cnt[ord(y) - 97] -= 1
    d = 0
    pos = neg = False
    for k in range(26):
        d += cnt[k]
        if d > 0: pos = True
        if d < 0: neg = True
    return not (pos and neg)

for (s1, s2), want in examples:
    print(s1, s2, can_break_prefix(s1, s2), can_break_prefix(s1, s2) == want)

abc xya True True
abe acd False True
leetcodee interview True True


In [13]:
class SolutionPrefix:
    def checkIfCanBreak(self, s1: str, s2: str) -> bool:
        cnt = Counter(s1)
        cnt.subtract(Counter(s2))
        d = 0
        pos = neg = False
        for ch in "abcdefghijklmnopqrstuvwxyz":
            d += cnt[ch]
            if d > 0:
                pos = True
            elif d < 0:
                neg = True
        return not (pos and neg)

print([SolutionPrefix().checkIfCanBreak(*p) for p, _ in examples])

[True, False, True]


**Độ phức tạp:** `O(n + 26)` thời gian, `O(26)` bộ nhớ phụ — tối ưu cả hai mặt.

## 8. TRACE từng bước

Trace cách 2 với ví dụ 3: `a = sorted("leetcodee") = cdeeeelot`, `b = sorted("interview") = eeiinrtvw`.

| i | a[i] | b[i] | a≥b? | b≥a? |
|---|---|---|---|---|
| 0 | c | e | ✘ | ✔ |
| 1 | d | e | ✘ | ✔ |
| 2 | e | i | ✘ | ✔ |
| 3 | e | i | ✘ | ✔ |
| 4 | e | n | ✘ | ✔ |
| 5 | e | r | ✘ | ✔ |
| 6 | l | t | ✘ | ✔ |
| 7 | o | v | ✘ | ✔ |
| 8 | t | w | ✘ | ✔ |

⇒ `s2` break `s1` ⇒ `true`.

In [14]:
def trace_sort(s1, s2):
    a, b = sorted(s1), sorted(s2)
    ge = le = True
    print(f"a = {''.join(a)}, b = {''.join(b)}")
    for i, (x, y) in enumerate(zip(a, b)):
        ge &= x >= y
        le &= y >= x
        print(f"i={i}: a[i]={x} b[i]={y} | còn có thể s1 break s2? {ge} | còn có thể s2 break s1? {le}")
    print("=> kết quả:", ge or le)
    return ge or le

trace_sort("leetcodee", "interview")
print()
trace_sort("abe", "acd")

a = cdeeeelot, b = eeiinrtvw
i=0: a[i]=c b[i]=e | còn có thể s1 break s2? False | còn có thể s2 break s1? True
i=1: a[i]=d b[i]=e | còn có thể s1 break s2? False | còn có thể s2 break s1? True
i=2: a[i]=e b[i]=i | còn có thể s1 break s2? False | còn có thể s2 break s1? True
i=3: a[i]=e b[i]=i | còn có thể s1 break s2? False | còn có thể s2 break s1? True
i=4: a[i]=e b[i]=n | còn có thể s1 break s2? False | còn có thể s2 break s1? True
i=5: a[i]=e b[i]=r | còn có thể s1 break s2? False | còn có thể s2 break s1? True
i=6: a[i]=l b[i]=t | còn có thể s1 break s2? False | còn có thể s2 break s1? True
i=7: a[i]=o b[i]=v | còn có thể s1 break s2? False | còn có thể s2 break s1? True
i=8: a[i]=t b[i]=w | còn có thể s1 break s2? False | còn có thể s2 break s1? True
=> kết quả: True

a = abe, b = acd
i=0: a[i]=a b[i]=a | còn có thể s1 break s2? True | còn có thể s2 break s1? True
i=1: a[i]=b b[i]=c | còn có thể s1 break s2? False | còn có thể s2 break s1? True
i=2: a[i]=e b[i]=d | còn có thể s1 

False

Trace cách 4 (tổng tiền tố) với `s1 = "abe"`, `s2 = "acd"` — chỉ in các chữ có thay đổi:

In [15]:
def trace_prefix(s1, s2):
    c1, c2 = count26(s1), count26(s2)
    d = 0; pos = neg = False
    for k in range(26):
        if c1[k] == 0 and c2[k] == 0:
            continue
        d += c1[k] - c2[k]
        pos |= d > 0; neg |= d < 0
        ch = chr(97 + k)
        print(f"chữ '{ch}': cnt1={c1[k]} cnt2={c2[k]} | d = (#≤'{ch}' trong s1) - (#≤'{ch}' trong s2) = {d:+d} | pos={pos} neg={neg}")
    print("=> kết quả:", not (pos and neg))

trace_prefix("abe", "acd")
print()
trace_prefix("abc", "xya")

chữ 'a': cnt1=1 cnt2=1 | d = (#≤'a' trong s1) - (#≤'a' trong s2) = +0 | pos=False neg=False
chữ 'b': cnt1=1 cnt2=0 | d = (#≤'b' trong s1) - (#≤'b' trong s2) = +1 | pos=True neg=False
chữ 'c': cnt1=0 cnt2=1 | d = (#≤'c' trong s1) - (#≤'c' trong s2) = +0 | pos=True neg=False
chữ 'd': cnt1=0 cnt2=1 | d = (#≤'d' trong s1) - (#≤'d' trong s2) = -1 | pos=True neg=True
chữ 'e': cnt1=1 cnt2=0 | d = (#≤'e' trong s1) - (#≤'e' trong s2) = +0 | pos=True neg=True
=> kết quả: False

chữ 'a': cnt1=1 cnt2=1 | d = (#≤'a' trong s1) - (#≤'a' trong s2) = +0 | pos=False neg=False
chữ 'b': cnt1=1 cnt2=0 | d = (#≤'b' trong s1) - (#≤'b' trong s2) = +1 | pos=True neg=False
chữ 'c': cnt1=1 cnt2=0 | d = (#≤'c' trong s1) - (#≤'c' trong s2) = +2 | pos=True neg=False
chữ 'x': cnt1=0 cnt2=1 | d = (#≤'x' trong s1) - (#≤'x' trong s2) = +1 | pos=True neg=False
chữ 'y': cnt1=0 cnt2=1 | d = (#≤'y' trong s1) - (#≤'y' trong s2) = +0 | pos=True neg=False
=> kết quả: True


## 9. Code nộp LeetCode

Dùng cách 4 (tổng tiền tố tần suất) — `O(n + 26)`:

In [16]:
class Solution:
    def checkIfCanBreak(self, s1: str, s2: str) -> bool:
        cnt = [0] * 26
        for x, y in zip(s1, s2):
            cnt[ord(x) - 97] += 1
            cnt[ord(y) - 97] -= 1
        d = 0                    # d = (#ký tự <= c trong s1) - (#ký tự <= c trong s2)
        pos = neg = False
        for k in range(26):
            d += cnt[k]
            if d > 0:
                pos = True       # s1 có nhiều ký tự nhỏ hơn -> s1 không break được s2
            elif d < 0:
                neg = True       # s2 có nhiều ký tự nhỏ hơn -> s2 không break được s1
        return not (pos and neg)

for (s1, s2), want in examples:
    assert Solution().checkIfCanBreak(s1, s2) == want
print("3 ví dụ: OK")

3 ví dụ: OK


## 10. Vì sao thuật toán đúng

**Bổ đề (sắp xếp tối ưu):** `s1` break `s2` (sau khi hoán vị) **⇔** `a[i] ≥ b[i]` với mọi `i`, trong đó `a = sorted(s1)`, `b = sorted(s2)`.

- (⇐) Chọn chính `a` và `b` làm hai hoán vị.
- (⇒) Giả sử có cách ghép mỗi ký tự `s1` với một ký tự `s2` mà bên `s1` luôn `≥`. Xét `i` bất kỳ. Có `n − i` ký tự của `s1` lớn hơn hoặc bằng… cụ thể: `i + 1` ký tự **nhỏ nhất** của `s1` là `a[0..i]`, tất cả `≤ a[i]`. Đối tác của chúng trong `s2` là `i + 1` ký tự **khác nhau**, mỗi cái `≤` đối tác `s1` của nó `≤ a[i]`. Vậy `s2` có ít nhất `i + 1` ký tự `≤ a[i]` ⇒ ký tự nhỏ thứ `i + 1` của `s2` là `b[i] ≤ a[i]` ✔.

**Tương đương tần suất (cách 4):** `a[i] ≥ b[i]` ∀`i` ⇔ với mọi chữ `c`: `#{s1 ≤ c} ≤ #{s2 ≤ c}`. Thật vậy, `#{s1 ≤ c} = |{i : a[i] ≤ c}|`; nếu `a[i] ≥ b[i]` thì `a[i] ≤ c ⇒ b[i] ≤ c` ⇒ bất đẳng thức đúng. Ngược lại nếu tồn tại `i` với `a[i] < b[i]`, lấy `c = a[i]`: `#{s1 ≤ c} ≥ i + 1` còn `#{s2 ≤ c} ≤ i` (vì `b[i] > c` và `b` tăng) ⇒ vi phạm.

Do đó "`s1` break `s2`" ⇔ mọi `d ≤ 0` (không có `pos`), "`s2` break `s1`" ⇔ mọi `d ≥ 0` (không có `neg`). Đáp án `= not pos or not neg = not (pos and neg)` ✔.

## 11. Phân tích Time / Space

Gọi `n = len(s1)`, `σ = 26` (kích thước bảng chữ).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute hoán vị | `O(n! · n)` | `O(n! · n)` (set) | chỉ `n ≤ 7` |
| 2. Sort + so | `O(n log n)` | `O(n)` | ngắn nhất |
| 3. Counting sort + so | `O(n + σ)` | `O(n)` | một lượt, dừng sớm |
| 4. Tổng tiền tố tần suất | `O(n + σ)` | `O(σ)` | tối ưu |

In [17]:
rng = random.Random(1433)
n = 10**5
s1 = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(n))
s2 = "".join(sorted(s1))[::-1]                       # một hoán vị của s1 (luôn true)
s3 = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(n))
for name, f in [("sort", can_break_sort), ("counting", can_break_counting_sort), ("prefix", can_break_prefix)]:
    t = time.perf_counter(); r1 = f(s1, s2); r2 = f(s1, s3); el = time.perf_counter() - t
    print(f"{name:9s}: n=10^5 x 2 lần, kết quả ({r1}, {r2}), {el:.3f}s")

sort     : n=10^5 x 2 lần, kết quả (True, False), 0.041s


counting : n=10^5 x 2 lần, kết quả (True, False), 0.021s


prefix   : n=10^5 x 2 lần, kết quả (True, False), 0.019s


## 12. Bộ test

1. Ba ví dụ đề.
2. Edge: `n = 1`; hai chuỗi giống nhau (`≥` nên `true`); một chuỗi là hoán vị của chuỗi kia (`true`); toàn `'a'` vs toàn `'z'`; trường hợp "đan xen" (`false`).
3. Đối chiếu với brute force hoán vị trên `n ≤ 6` (bảng chữ nhỏ để dễ có cả `true` lẫn `false`).
4. Ngẫu nhiên lớn hơn giữa các cách nhanh; kiểm tra tính đối xứng `f(s1, s2) == f(s2, s1)`.

In [18]:
impls = {
    "brute": can_break_brute, "sort": can_break_sort, "counting": can_break_counting_sort, "prefix": can_break_prefix,
    "Solution": lambda a, b: Solution().checkIfCanBreak(a, b),
    "SolSort": lambda a, b: SolutionSort().checkIfCanBreak(a, b),
    "SolPrefix": lambda a, b: SolutionPrefix().checkIfCanBreak(a, b),
}
edge = [(("a", "z"), True), (("q", "q"), True), (("abc", "cab"), True), (("aaaa", "zzzz"), True),
        (("az", "by"), False), (("ac", "bb"), False), (("abcd", "bbbb"), False), (("zza", "yyb"), False)]
for name, f in impls.items():
    if name == "brute":
        cases = [c for c in examples + edge if len(c[0][0]) <= 7]
    else:
        cases = examples + edge
    for (a, b), want in cases:
        assert f(a, b) == want, (name, a, b)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [19]:
rng = random.Random(33)
stats = Counter()
for _ in range(600):
    n = rng.randint(1, 6)
    alpha = rng.choice(["ab", "abc", "abcd"])
    a = "".join(rng.choice(alpha) for _ in range(n))
    b = "".join(rng.choice(alpha) for _ in range(n))
    want = can_break_brute(a, b)
    stats[want] += 1
    for name, f in impls.items():
        assert f(a, b) == want, (name, a, b)
        assert f(b, a) == want
print("đối chiếu brute 600 cặp:", dict(stats))

for _ in range(300):
    n = rng.randint(1, 500)
    a = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(n))
    b = "".join(rng.choice("abcdefghij") for _ in range(n))
    r = can_break_sort(a, b)
    assert all(f(a, b) == r for nm, f in impls.items() if nm != "brute")
print("random n ≤ 500: OK")

đối chiếu brute 600 cặp: {True: 559, False: 41}


random n ≤ 500: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra một chiều

In [20]:
def wrong_one_way(s1, s2):
    a, b = sorted(s1), sorted(s2)
    return all(x >= y for x, y in zip(a, b))       # SAI: quên chiều s2 break s1

for s1, s2 in [("abc", "xya"), ("leetcodee", "interview")]:
    print(s1, s2, "| sai:", wrong_one_way(s1, s2), "| đúng:", Solution().checkIfCanBreak(s1, s2))

abc xya | sai: False | đúng: True
leetcodee interview | sai: False | đúng: True


**Vì sao sai:** đề hỏi "`s1` break `s2` **hoặc ngược lại**". Ví dụ 1 chỉ đúng theo chiều `s2` break `s1`.

**Sửa:** `breaks(a, b) or breaks(b, a)`, hoặc theo dõi cả `pos` và `neg`.

### Lỗi 2 — So sánh khi chưa sắp xếp

In [21]:
def wrong_no_sort(s1, s2):
    return all(x >= y for x, y in zip(s1, s2)) or all(y >= x for x, y in zip(s1, s2))   # SAI

for s1, s2 in [("abc", "xya"), ("ba", "ab")]:
    print(s1, s2, "| sai:", wrong_no_sort(s1, s2), "| đúng:", Solution().checkIfCanBreak(s1, s2))

abc xya | sai: False | đúng: True
ba ab | sai: False | đúng: True


**Vì sao sai:** đề cho phép **hoán vị**; thứ tự gốc không có ý nghĩa. `"ba"` và `"ab"` là hoán vị của nhau nên chắc chắn `true`, nhưng so trực tiếp thì `b ≥ a` nhưng `a ≥ b` sai (và ngược lại).

**Sửa:** sort cả hai trước (hoặc dùng tần suất).

### Lỗi 3 — Cho phép đổi chiều ở từng vị trí

In [22]:
def wrong_mixed(s1, s2):
    a, b = sorted(s1), sorted(s2)
    return all(x >= y or y >= x for x, y in zip(a, b))   # SAI: luôn True

for s1, s2 in [("abe", "acd"), ("az", "by")]:
    print(s1, s2, "| sai:", wrong_mixed(s1, s2), "| đúng:", Solution().checkIfCanBreak(s1, s2))

abe acd | sai: True | đúng: False
az by | sai: True | đúng: False


**Vì sao sai:** chiều phải **cố định cho toàn bộ chuỗi** (một chuỗi `≥` chuỗi kia ở **mọi** vị trí). `x ≥ y or y ≥ x` luôn đúng với mọi cặp ký tự ⇒ hàm luôn trả `True`.

**Sửa:** tính `all(...)` cho từng chiều riêng rồi mới `or`.

### Lỗi 4 — Dùng `>` thay cho `>=`

In [23]:
def wrong_strict(s1, s2):
    a, b = sorted(s1), sorted(s2)
    return all(x > y for x, y in zip(a, b)) or all(y > x for x, y in zip(a, b))   # SAI

for s1, s2 in [("abc", "abc"), ("abc", "xya")]:
    print(s1, s2, "| sai:", wrong_strict(s1, s2), "| đúng:", Solution().checkIfCanBreak(s1, s2))

abc abc | sai: False | đúng: True
abc xya | sai: False | đúng: True


**Vì sao sai:** định nghĩa là `x[i] >= y[i]` — bằng nhau vẫn được. Hai chuỗi giống hệt nhau luôn break nhau; ví dụ 1 có cặp `a ≥ a`.

**Sửa:** dùng `>=`. Ở cách 4, điều này ứng với `d = 0` **không** bị coi là vi phạm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Được **hoán vị tuỳ ý**, cần ghép cặp sao cho mọi cặp thoả bất đẳng thức" ⇒ **sort cả hai và ghép theo thứ tự** (lập luận trao đổi). Giống **870 Advantage Shuffle**, **455 Assign Cookies**, **2410**.
- Bảng chữ cái nhỏ (26) ⇒ đếm tần suất thay cho sort: `O(n + 26)`.
- "Dominance" giữa hai multiset ⇔ so sánh **tổng tiền tố tần suất** (hàm phân phối tích luỹ).
- Câu hỏi "hoặc ngược lại" ⇒ kiểm tra **hai chiều**; một lượt với hai cờ là đủ.
- Test nhanh: hai chuỗi bằng nhau (`true`), hoán vị của nhau (`true`), `"az"/"by"` (`false`).

## 15. Biến thể của bài

1. **Chỉ một chiều** (`s1` break `s2`?) ⇒ bỏ cờ còn lại.
2. **Số thay đổi ít nhất** trong `s1` để `s1` break được `s2` ⇒ ghép tối đa cặp `a ≥ b` bằng hai con trỏ, phần còn lại đổi thành `'z'`.
3. **LeetCode 870 Advantage Shuffle**: tối đa số vị trí `a[i] > b[i]` (b cố định thứ tự) ⇒ sort + tham lam (điền "điền thế" phần tử nhỏ nhất).
4. **Mảng số nguyên** thay vì chuỗi ⇒ dùng sort (không có counting 26).
5. **Ba chuỗi** — có tồn tại thứ tự `x ≥ y ≥ z` sau hoán vị? ⇒ sort cả ba, kiểm tra các cặp theo một trong 6 thứ tự.

Code biến thể 3 (870 Advantage Shuffle — đếm số vị trí thắng tối đa):

In [24]:
def max_wins(A, B):
    A, B = sorted(A), sorted(B)
    j = wins = 0
    for x in A:                 # dùng phần tử nhỏ nhất của A có thể thắng B[j]
        if x > B[j]:
            wins += 1
            j += 1
    return wins

print(max_wins([2, 7, 11, 15], [1, 10, 4, 11]), max_wins([12, 24, 8, 32], [13, 25, 32, 11]))

4 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute hoán vị | Thử mọi hoán vị của `s2` | `O(n!·n)` | `O(n!·n)` | Kiểm chứng `n ≤ 7` |
| 2. Sort + so | Ghép nhỏ–nhỏ, kiểm tra hai chiều | `O(n log n)` | `O(n)` | Mặc định, dễ viết |
| 3. Counting sort + so | Dựng chuỗi đã sắp từ 26 bộ đếm | `O(n + 26)` | `O(n)` | Muốn `O(n)` vẫn trực quan |
| 4. Tổng tiền tố tần suất | `d = pre1 − pre2` không đổi dấu | `O(n + 26)` | `O(26)` | Tối ưu, chứng minh đẹp |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().checkIfCanBreak("abc", "bcd")` và `checkIfCanBreak("adz", "bbb")` trả gì?

### Bài 2 (tính tay)
Sort hai chuỗi của ví dụ 3 bằng tay và so từng vị trí; sau đó tính dãy `d` của cách 4 tại các chữ `c, d, e, i, l, n, o, r, t, v, w`.

### Bài 3 (điền TODO)
Hoàn thiện `can_break_one_way(x, y)`: `True` nếu **`x` break `y`** (một chiều), dùng tổng tiền tố tần suất.

### Bài 4 (tự viết lại)
Viết `break_direction(s1, s2)` trả về `"both"`, `"s1"` (chỉ s1 break s2), `"s2"` hoặc `"none"`.

### Bài 5 (mini challenge)
Viết `min_changes_to_break(s1, s2)`: số ký tự **ít nhất** cần thay đổi trong `s1` (đổi thành chữ bất kỳ) để `s1` break được `s2` (một chiều).

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def can_break_one_way(x, y):
    cx, cy = count26(x), count26(y)
    # TODO: duyệt k, cộng dồn d += cx[k] - cy[k]; nếu d > 0 thì x có quá nhiều ký tự nhỏ -> False
    pass

# Bài 4 — TODO
def break_direction(s1, s2):
    # TODO: tính pos/neg như cách 4 rồi ánh xạ ra 4 trường hợp
    pass

# Bài 5 — TODO
def min_changes_to_break(s1, s2):
    # TODO: sort; ghép tối đa cặp a >= b bằng hai con trỏ; đáp án = n - số cặp
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `abc` vs `bcd`: sorted `bcd ≥ abc` từng vị trí ⇒ `true`. (2) `adz` vs `bbb`: `a < b` nhưng `d > b` ⇒ không chiều nào ⇒ `false`. (3) Kiểm tra lại bằng cách 4: `d` tại `a` là `+1`, tại `b` là `−2`.

**Bài 2.** (1) `cdeeeelot` vs `eeiinrtvw`. (2) Mọi vị trí `b[i] ≥ a[i]` ⇒ `true`. (3) `d` lần lượt: `c:1, d:2, e:4, i:2, l:3, n:2, o:3, r:2, t:2, v:1, w:0` — **không bao giờ âm** ⇒ `s2` break `s1`.

**Bài 3.** (1) `x` break `y` ⇔ mọi `#x≤c ≤ #y≤c`. (2) `d = pre_x − pre_y`; `d > 0` ⇒ `False`. (3) Hết vòng ⇒ `True`.

**Bài 4.** (1) `s1` break `s2` ⇔ không `pos`; `s2` break `s1` ⇔ không `neg`. (2) Cả hai ⇒ `"both"` (khi và chỉ khi hai multiset bằng nhau). (3) Còn lại ánh xạ trực tiếp.

**Bài 5.** (1) Ký tự bị đổi nên đổi thành `'z'` — luôn thắng. (2) Giữ lại được nhiều nhất bao nhiêu ký tự? = số cặp tối đa `a ≥ b` với các cặp rời nhau: sort, duyệt `a` tăng dần, nếu `a ≥ b[j]` thì ghép và `j += 1`. (3) Đáp án `n − số cặp`; kiểm chứng bằng vét cạn mọi tập vị trí đổi thành `'z'`.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert Solution().checkIfCanBreak("abc", "bcd") is True
assert Solution().checkIfCanBreak("adz", "bbb") is False

# Bài 2
assert "".join(sorted("leetcodee")) == "cdeeeelot" and "".join(sorted("interview")) == "eeiinrtvw"
c1, c2 = count26("leetcodee"), count26("interview")
ds, d = {}, 0
for k in range(26):
    d += c1[k] - c2[k]
    if c1[k] or c2[k]:
        ds[chr(97 + k)] = d
print("dãy d của ví dụ 3:", ds)
assert all(v >= 0 for v in ds.values())

# Bài 3
def can_break_one_way_sol(x, y):
    cx, cy = count26(x), count26(y)
    d = 0
    for k in range(26):
        d += cx[k] - cy[k]
        if d > 0:
            return False
    return True

# Bài 4
def break_direction_sol(s1, s2):
    one, two = can_break_one_way_sol(s1, s2), can_break_one_way_sol(s2, s1)
    return {(True, True): "both", (True, False): "s1", (False, True): "s2", (False, False): "none"}[(one, two)]

assert break_direction_sol("abc", "cab") == "both"
assert break_direction_sol("abc", "xya") == "s2"
assert break_direction_sol("zzz", "abc") == "s1"
assert break_direction_sol("abe", "acd") == "none"

# Bài 5
def min_changes_to_break_sol(s1, s2):
    a, b = sorted(s1), sorted(s2)
    j = pairs = 0
    for x in a:
        if j < len(b) and x >= b[j]:
            pairs += 1
            j += 1
    return len(s1) - pairs

def brute_min_changes(s1, s2):
    n = len(s1)
    best = n
    for mask in range(1 << n):
        t = "".join("z" if mask >> i & 1 else s1[i] for i in range(n))
        if can_break_one_way_sol(t, s2):
            best = min(best, bin(mask).count("1"))
    return best

assert min_changes_to_break_sol("abe", "acd") == 1
rng = random.Random(4)
for _ in range(400):
    n = rng.randint(1, 7)
    a = "".join(rng.choice("abcd") for _ in range(n))
    b = "".join(rng.choice("abcd") for _ in range(n))
    assert can_break_one_way_sol(a, b) == any(breaks(a, p) for p in permutations(b))
    assert (break_direction_sol(a, b) != "none") == can_break_brute(a, b)
    assert min_changes_to_break_sol(a, b) == brute_min_changes(a, b)
print("Tất cả đáp án bài tập: OK")

dãy d của ví dụ 3: {'c': 1, 'd': 2, 'e': 4, 'i': 2, 'l': 3, 'n': 2, 'o': 3, 'r': 2, 't': 2, 'v': 1, 'w': 0}


Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template sort + hai chiều:**

```python
a, b = sorted(s1), sorted(s2)
return all(x >= y for x, y in zip(a, b)) or all(y >= x for x, y in zip(a, b))
```

**Template tần suất 26 chữ (dominance):**

```python
cnt = [0] * 26
for x, y in zip(s1, s2):
    cnt[ord(x) - 97] += 1
    cnt[ord(y) - 97] -= 1
d, pos, neg = 0, False, False
for k in range(26):
    d += cnt[k]
    pos |= d > 0
    neg |= d < 0
return not (pos and neg)
```

**Checklist:**
- [ ] Đã sort (hoặc dùng tần suất) — không so thứ tự gốc.
- [ ] Kiểm tra **cả hai chiều**.
- [ ] Chiều cố định cho cả chuỗi, không đổi chiều theo từng vị trí.
- [ ] Dùng `>=` (bằng nhau vẫn break).
- [ ] Đã test chuỗi bằng nhau, hoán vị của nhau, `"az"/"by"`, `n = 1`.